# Notebook 15 — LoRA Edit B Control: Does In-Place Modification Cause Damage?
**CS 590NN | Amogh | Apr 25**

## What we're testing

NB11 found the optimizer step on B's circuit destroys A regardless of B's content. NB12 found ACDC-circuit overlap doesn't predict it (ρ = -0.28).

Open question: **is the damage from in-place modification of original parameters, or from gradient flow through the network during optimization?**

LoRA gives us a clean test. A LoRA edit:
- Freezes the original W_in, W_out
- Adds a low-rank adapter B@A added to outputs
- Optimizer.step() updates only the adapter, NOT the original weights

If LoRA edits don't damage A while direct edits do → **in-place parameter modification is the cause**. The "optimizer step" mechanism from NB11 is more specifically "optimizer step that overwrites original weights."

If LoRA edits DO damage A → the damage propagates through the network during forward/backward passes regardless of where the trainable parameters live. Mechanism is more diffuse than direct parameter overwrite.

## Implementation note

Standard `peft.LoraConfig` wraps `nn.Linear` modules. TransformerLens stores MLPs as raw `W_in` / `W_out` tensors, so we implement LoRA manually as a forward hook on `hook_mlp_out`.

## Compute

~15 min on A100. 10 pairs × 3 conditions = 30 runs.

### 15.0 Install (run once, restarts)

In [ ]:
import subprocess, sys, os
def pip(args): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + args)
pip(['numpy==1.26.4'])
pip(['transformer-lens', 'transformers', 'datasets', 'accelerate', 'einops', 'jaxtyping'])
pip(['matplotlib', 'scipy'])
print('Restarting...'); os.kill(os.getpid(), 9)

### 15.1 Imports + auto-fetch v3 pair indices

In [ ]:
import torch, json, requests, random
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from scipy import stats
from transformer_lens import HookedTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = Path('results_nb15'); RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = RESULTS_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)
torch.manual_seed(42); random.seed(42); np.random.seed(42)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {torch.cuda.get_device_name(0)} ({free/1e9:.1f}/{total/1e9:.1f} GB free)')

REPO_RAW = 'https://raw.githubusercontent.com/rukmini-17/targeted-unlearning/main/circuit_pipeline/results'
V3_URL = f'{REPO_RAW}/sequential_editing_v3/sequential_edit_rows_v3_20260424_220951.json'
print(f'\nFetching v3 pair indices from GitHub...')
v3_data = requests.get(V3_URL, timeout=60).json()
v3_ok = [r for r in v3_data['rows'] if r.get('status') == 'ok']
seen = set()
PAIR_INDICES = []
for r in v3_ok:
    pid = r['pair_idx']
    if pid in seen: continue
    seen.add(pid)
    PAIR_INDICES.append((r['A_idx'], r['B_idx']))
    if len(PAIR_INDICES) >= 10: break
print(f'Selected {len(PAIR_INDICES)} pair indices from v3')

### 15.2 Load model

In [ ]:
MODEL_NAME = 'Qwen/Qwen3-0.6B'
model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    center_unembed=True, center_writing_weights=False,
    fold_ln=True, refactor_factored_attn_matrices=False, device=DEVICE,
)
model.set_use_attn_result(True)
model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token
print(f'Loaded {MODEL_NAME}: {model.cfg.n_layers}L x {model.cfg.n_heads}H, d_model={model.cfg.d_model}')

### 15.3 Load CounterFact

In [ ]:
@dataclass
class EditSample:
    idx: int; prompt: str; subject: str
    target_new: str; target_true: str

raw = requests.get('https://rome.baulab.info/data/dsets/counterfact.json', timeout=120).json()
def make_sample(idx):
    rr = raw[idx]['requested_rewrite']
    return EditSample(idx=idx, prompt=rr['prompt'].format(rr['subject']),
                      subject=rr['subject'],
                      target_new=' '+rr['target_new']['str'], target_true=' '+rr['target_true']['str'])

pairs = [{'A': make_sample(a), 'B': make_sample(b)} for a, b in PAIR_INDICES]
print(f'Built {len(pairs)} pairs')

### 15.4 ROME-trace localization (same as NB11)

In [ ]:
def rome_trace_top_layers(model, sample, k=5):
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0].item()
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0,0].item()
    tokens = model.to_tokens(sample.prompt)
    n_tok = tokens.shape[1]
    if n_tok <= 2: return list(range(min(k, model.cfg.n_layers)))
    subj_pos = list(range(1, n_tok-1))
    corrupt = tokens.clone()
    corrupt[0, subj_pos] = torch.randint(1000, model.cfg.d_vocab-1, (len(subj_pos),), device=tokens.device)
    with torch.no_grad():
        cl, cc = model.run_with_cache(tokens)
        kl, _ = model.run_with_cache(corrupt)
    clean_score = (cl[0,-1,true_id] - cl[0,-1,new_id]).item()
    corrupt_score = (kl[0,-1,true_id] - kl[0,-1,new_id]).item()
    eff = max(abs(clean_score - corrupt_score), 0.5)
    effects = []
    for L in range(model.cfg.n_layers):
        hn = f'blocks.{L}.hook_mlp_out'
        if hn not in cc: continue
        ca = cc[hn].clone()
        def hk(v, hook): return ca
        with torch.no_grad():
            patched = model.run_with_hooks(corrupt, fwd_hooks=[(hn, hk)])
        recovery = (patched[0,-1,true_id] - patched[0,-1,new_id]).item() - corrupt_score
        effects.append((L, abs(recovery)/eff))
    effects.sort(key=lambda x: -x[1])
    del cc; torch.cuda.empty_cache()
    return [L for L, _ in effects[:k]]

### 15.5 Manual LoRA implementation for TransformerLens

For each MLP layer in our edit set, register a forward hook on `hook_mlp_out` that adds a low-rank delta:

```
mlp_out_modified = mlp_out + (B @ A)(mlp_out)
```

A: (d_model, r), B: (r, d_model), r=8. Only A and B are trainable. The original W_in, W_out are frozen.

In [ ]:
class LoRAAdapter(nn.Module):
    """Low-rank adapter added to MLP output via TransformerLens hook."""
    def __init__(self, d_model, rank=8):
        super().__init__()
        self.A = nn.Parameter(torch.randn(d_model, rank, device=DEVICE) * 0.01)
        self.B = nn.Parameter(torch.zeros(rank, d_model, device=DEVICE))
    def forward(self, x):
        # x: (..., d_model) -> add (x @ A) @ B
        return x + (x @ self.A) @ self.B

def setup_lora_adapters(model, mlp_layers, rank=8):
    """Create LoRA adapters and register hooks on hook_mlp_out for given layers."""
    adapters = {}
    handles = []
    for L in mlp_layers:
        adapter = LoRAAdapter(model.cfg.d_model, rank=rank).to(DEVICE)
        adapters[L] = adapter
        hook_name = f'blocks.{L}.hook_mlp_out'
        def make_hook(adapter):
            def hook(value, hook):
                return adapter(value)
            return hook
        handle = model.add_hook(hook_name, make_hook(adapter))
        handles.append(handle)
    return adapters, handles

def remove_lora_adapters(model):
    """Clear all hooks (resets the LoRA additions)."""
    model.reset_hooks()

# Quick smoke test
test_layers = [10, 12]
adapters, _ = setup_lora_adapters(model, test_layers, rank=8)
print(f'Created {len(adapters)} LoRA adapters')
print(f'  Each adapter has {sum(p.numel() for p in adapters[10].parameters())} params')
print(f'  Trainable params total: {sum(sum(p.numel() for p in a.parameters()) for a in adapters.values())}')
remove_lora_adapters(model)
print('Cleared.')

### 15.6 Edit functions — full and LoRA variants

In [ ]:
NEUTRAL = [
    'The sum of two and three is', 'Twelve divided by four equals',
    'The square root of nine is', 'Ten times ten equals',
    'The capital of Japan is', 'The largest ocean on Earth is the',
    'Mount Everest is located in the', 'The Amazon River flows through',
    'Water boils at one hundred degrees', 'The chemical symbol for gold is',
    'Plants produce oxygen through a process called', 'The Earth orbits the',
    'A week contains seven', 'The primary colors are red, blue, and',
    'Humans have two lungs and one', 'A triangle has three',
]

def cache_ref(model, prompts):
    cache = []
    model.eval()
    with torch.no_grad():
        for p in prompts:
            t = model.to_tokens(p)
            ref_lp = torch.log_softmax(model(t)[0,-1,:], dim=-1).detach().clone()
            cache.append((t, ref_lp))
    return cache

def kl_against(model, ref_cache):
    total = 0.0
    for t, ref_lp in ref_cache:
        edit_lp = torch.nn.functional.log_softmax(model(t)[0,-1,:], dim=-1)
        total = total + (ref_lp.exp() * (ref_lp - edit_lp)).sum()
    return total / len(ref_cache)

def edit_full(model, sample, mlp_layers, n_steps, lr, beta_kl, ref_cache):
    """NB11-style: modify W_in, W_out in place via Adam."""
    params = []
    for L in mlp_layers:
        params += [model.blocks[L].mlp.W_in, model.blocks[L].mlp.W_out]
    if not params:
        for L in range(model.cfg.n_layers):
            params += [model.blocks[L].mlp.W_in, model.blocks[L].mlp.W_out]
    for p in model.parameters(): p.requires_grad_(False)
    for p in params: p.requires_grad_(True)
    opt = torch.optim.Adam(params, lr=lr)
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0,0]
    tokens = model.to_tokens(sample.prompt)
    for _ in range(n_steps):
        model.train()
        opt.zero_grad(set_to_none=True)
        logits = model(tokens)
        lp = torch.nn.functional.log_softmax(logits[0,-1,:], dim=-1)
        loss = -lp[new_id] + lp[true_id]
        if ref_cache and beta_kl > 0:
            loss = loss + beta_kl * kl_against(model, ref_cache)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
        opt.step()
    for p in model.parameters(): p.requires_grad_(True)
    torch.cuda.empty_cache()

def edit_lora(model, sample, mlp_layers, n_steps, lr, beta_kl, ref_cache, rank=8):
    """LoRA-style: freeze W_in, W_out. Add LoRA adapter via hook. Train ONLY adapter."""
    if not mlp_layers:
        mlp_layers = list(range(model.cfg.n_layers))
    
    # Set up adapters and hooks (these become part of the forward pass)
    adapters, _ = setup_lora_adapters(model, mlp_layers, rank=rank)
    
    # Freeze ALL original model params
    for p in model.parameters(): p.requires_grad_(False)
    # Only adapters are trainable
    adapter_params = []
    for adapter in adapters.values():
        for p in adapter.parameters():
            p.requires_grad_(True)
            adapter_params.append(p)
    
    opt = torch.optim.Adam(adapter_params, lr=lr)
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0,0]
    tokens = model.to_tokens(sample.prompt)
    
    for _ in range(n_steps):
        model.train()
        opt.zero_grad(set_to_none=True)
        logits = model(tokens)
        lp = torch.nn.functional.log_softmax(logits[0,-1,:], dim=-1)
        loss = -lp[new_id] + lp[true_id]
        if ref_cache and beta_kl > 0:
            loss = loss + beta_kl * kl_against(model, ref_cache)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(adapter_params, max_norm=1.0)
        opt.step()
    
    # Restore grad state on original params (don't remove the LoRA hook yet — 
    # measurements happen with LoRA active)
    for p in model.parameters(): p.requires_grad_(True)
    torch.cuda.empty_cache()
    return adapters  # return for later cleanup

def measure_p_new(model, sample):
    model.eval()
    with torch.no_grad():
        new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
        return float(torch.softmax(model(model.to_tokens(sample.prompt))[0,-1,:], dim=-1)[new_id])

def restore(model, state):
    with torch.no_grad():
        for n, p in model.named_parameters(): p.copy_(state[n])
    model.reset_hooks()  # clear any LoRA hooks
    torch.cuda.empty_cache()

print('Snapshotting weights...')
original_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
torch.cuda.empty_cache()
free, _ = torch.cuda.mem_get_info()
print(f'GPU free: {free/1e9:.1f} GB')

### 15.7 The 3-condition loop

In [ ]:
CONDITIONS = ['skip', 'full', 'lora']  # C0, C1, C2
N_STEPS_A = 5
N_STEPS_B = 3
BETA_KL = 0.1
LR_FULL = 5e-3
LR_LORA = 1e-2  # LoRA usually wants higher lr since fewer params

ROWS_FILE = RESULTS_DIR / f'lora_control_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
rows = []
started = datetime.now()
ref_cache = cache_ref(model, NEUTRAL)

all_samples = {s.idx: s for p in pairs for s in (p['A'], p['B'])}
print(f'Pre-computing ROME-trace for {len(all_samples)} unique samples...')
circuits = {idx: rome_trace_top_layers(model, s, k=5) for idx, s in all_samples.items()}
print('Done.')

total_runs = len(pairs) * len(CONDITIONS)
print(f'Total runs: {total_runs}')

run_idx = 0
for p_idx, p in enumerate(pairs):
    A, B = p['A'], p['B']
    for cond in CONDITIONS:
        run_idx += 1
        try:
            # Reset and edit A (always full edit for A — only B varies)
            restore(model, original_state)
            edit_full(model, A, circuits[A.idx], n_steps=N_STEPS_A, lr=LR_FULL, beta_kl=BETA_KL,
                      ref_cache=ref_cache)
            p_A_postA = measure_p_new(model, A)
            
            # Apply condition for B
            adapters_to_clean = None
            if cond == 'skip':
                pass  # no edit B
            elif cond == 'full':
                edit_full(model, B, circuits[B.idx], n_steps=N_STEPS_B, lr=LR_FULL, beta_kl=BETA_KL,
                          ref_cache=ref_cache)
            elif cond == 'lora':
                adapters_to_clean = edit_lora(model, B, circuits[B.idx], n_steps=N_STEPS_B,
                                              lr=LR_LORA, beta_kl=BETA_KL, ref_cache=ref_cache, rank=8)
            
            # Measure A and B (LoRA hooks remain active during measurement for the 'lora' condition)
            p_A_after = measure_p_new(model, A)
            p_B_after = measure_p_new(model, B)
            A_displaced = max(0, p_A_postA - p_A_after)
            
            row = {
                'pair_idx': p_idx, 'A_idx': A.idx, 'B_idx': B.idx, 'condition': cond,
                'p_A_postA': p_A_postA, 'p_A_after_B': p_A_after, 'p_B_after_B': p_B_after,
                'A_displaced': A_displaced, 'status': 'ok',
            }
            rows.append(row)
            print(f'[{run_idx:>2}/{total_runs}] pair {p_idx:>2}  cond={cond:>5}  '
                  f'p_A_postA={p_A_postA:.3f}  p_A_after={p_A_after:.3f}  '
                  f'A_displaced={A_displaced:.3f}  p_B_after={p_B_after:.3f}')
            with open(ROWS_FILE, 'w') as f: json.dump({'rows': rows}, f, indent=2)
        except Exception as e:
            print(f'[{run_idx:>2}/{total_runs}] FAILED: {type(e).__name__}: {e}')
            import traceback; traceback.print_exc()
            rows.append({'pair_idx': p_idx, 'condition': cond, 'status': 'failed', 'error': str(e)[:200]})
            torch.cuda.empty_cache()
            model.reset_hooks()

elapsed = (datetime.now() - started).total_seconds() / 60
print(f'\nTotal runtime: {elapsed:.1f} min')
restore(model, original_state)

### 15.8 Aggregate and test

In [ ]:
df = pd.DataFrame([r for r in rows if r.get('status') == 'ok'])
print(f'OK rows: {len(df)}')

agg = df.groupby('condition').agg(
    n=('pair_idx','count'),
    A_displaced_mean=('A_displaced','mean'),
    A_displaced_median=('A_displaced','median'),
    A_displaced_std=('A_displaced','std'),
    p_B_after_mean=('p_B_after_B','mean'),
    p_B_after_median=('p_B_after_B','median'),
).round(3)
print('\nPer-condition aggregate:')
print(agg)

# Pairwise tests
print('\n=== Pairwise Wilcoxon (paired) ===')
from itertools import combinations
results = {}
for c1, c2 in combinations(CONDITIONS, 2):
    piv = df.pivot_table(index='pair_idx', columns='condition', values='A_displaced').dropna(subset=[c1, c2])
    if len(piv) >= 4:
        try:
            w, p = stats.wilcoxon(piv[c1], piv[c2])
            sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
            diff = piv[c1].mean() - piv[c2].mean()
            print(f'  {c1:>6} vs {c2:>6}: p={p:.4f} {sig}  (diff={diff:+.3f})')
            results[(c1,c2)] = {'p': float(p), 'diff': float(diff)}
        except ValueError as e:
            print(f'  {c1:>6} vs {c2:>6}: Wilcoxon undefined (identical values)')
            results[(c1,c2)] = {'p': 1.0, 'diff': float(piv[c1].mean() - piv[c2].mean())}

# Sanity: did LoRA actually install B?
print('\n=== Sanity: did each condition install B? ===')
for c in CONDITIONS:
    sub = df[df['condition']==c]
    n_installed = (sub['p_B_after_B'] > 0.5).sum()
    print(f'  {c:>6}: B installed in {n_installed}/{len(sub)} pairs (mean p_B = {sub["p_B_after_B"].mean():.3f})')

print('\n=== Verdict ===')
m = {c: df[df['condition']==c]['A_displaced'].mean() for c in CONDITIONS}
b = {c: df[df['condition']==c]['p_B_after_B'].mean() for c in CONDITIONS}
print(f'  skip:  A_displaced = {m["skip"]:.3f}  p_B_after = {b["skip"]:.3f}')
print(f'  full:  A_displaced = {m["full"]:.3f}  p_B_after = {b["full"]:.3f}')
print(f'  lora:  A_displaced = {m["lora"]:.3f}  p_B_after = {b["lora"]:.3f}')

if b['lora'] < 0.3:
    print('\n>>> CAVEAT: LoRA did not successfully install B. Comparison is uninformative.')
    print('    Try increasing LR_LORA, rank, or N_STEPS_B and rerun.')
elif m['lora'] < m['full'] * 0.3:
    print('\n>>> NOVEL: LoRA edits damage A much LESS than full edits even when B installs.')
    print('    Confirms: in-place modification of W_in/W_out is the cause, not just gradient flow.')
elif abs(m['lora'] - m['full']) < 0.15:
    print('\n>>> LoRA and full edits damage A similarly.')
    print('    Damage propagates through forward/backward passes regardless of where trainable params live.')
else:
    print('\n>>> Intermediate result. See per-condition table.')

### 15.9 Figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
order = ['skip','lora','full']
labels = ['C0: no edit B', 'C2: LoRA edit B', 'C1: full edit B']
means = [df[df['condition']==c]['A_displaced'].mean() for c in order]
stds  = [df[df['condition']==c]['A_displaced'].std()  for c in order]
colors = ['#888888', '#33aa33', '#cc3333']
bars = ax.bar(labels, means, yerr=stds, capsize=5, color=colors, edgecolor='black', alpha=0.85)
ax.set_ylabel('A_displaced')
ax.set_title('Does LoRA avoid the damage?')
ax.grid(alpha=0.3, axis='y'); ax.set_ylim(-0.05, 1.05)
for bar, mv in zip(bars, means):
    ax.annotate(f'{mv:.2f}', (bar.get_x()+bar.get_width()/2, bar.get_height()+0.02),
                ha='center', fontsize=10, fontweight='bold')

# Right — show p_B_after as sanity check
ax = axes[1]
b_means = [df[df['condition']==c]['p_B_after_B'].mean() for c in order]
b_stds  = [df[df['condition']==c]['p_B_after_B'].std()  for c in order]
ax.bar(labels, b_means, yerr=b_stds, capsize=5, color=colors, edgecolor='black', alpha=0.85)
ax.set_ylabel('p_B_after (did B install?)')
ax.set_title('Sanity: did each condition install B?')
ax.grid(alpha=0.3, axis='y'); ax.set_ylim(-0.05, 1.05)
for i, mv in enumerate(b_means):
    ax.annotate(f'{mv:.2f}', (i, mv+0.02), ha='center', fontsize=10, fontweight='bold')

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig1_lora_control.png', dpi=140)
plt.show()

### 15.10 Save

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
summary = {
    'notebook': 'Notebook 15 — LoRA Edit B Control',
    'timestamp': ts,
    'model': MODEL_NAME,
    'n_pairs': len(pairs),
    'conditions': CONDITIONS,
    'data_source': 'pair indices auto-fetched from github.com/rukmini-17/targeted-unlearning',
    'lora_config': {'rank': 8, 'lr': LR_LORA},
    'per_condition_means_A_displaced': {c: float(df[df['condition']==c]['A_displaced'].mean()) for c in CONDITIONS},
    'per_condition_means_p_B_after':   {c: float(df[df['condition']==c]['p_B_after_B'].mean())  for c in CONDITIONS},
    'per_condition_n': {c: int((df['condition']==c).sum()) for c in CONDITIONS},
    'pairwise_tests': {f'{a}_vs_{b}': v for (a,b), v in results.items()},
}
with open(RESULTS_DIR / f'summary_nb15_{ts}.json', 'w') as f: json.dump(summary, f, indent=2)
df.to_csv(RESULTS_DIR / f'df_nb15_{ts}.csv', index=False)
print(json.dumps(summary, indent=2))

import shutil
bundle = f'nb15_results_{ts}'
shutil.make_archive(bundle, 'zip', RESULTS_DIR)
from google.colab import files
files.download(f'{bundle}.zip')